In [12]:
import great_expectations as gx
import pandas as pd
import warnings
warnings.filterwarnings("ignore", message="`result_format` configured at the Validator-level*")
from gx_type_inference import apply_type_expectations, cast_dataframe_to_inferred_types

# Load the data
df = pd.read_csv("./data/transactions.csv", dtype=str)
#df = df.astype(str) # coerce everything to strings
#df = df.replace({pd.NA: None}) # change null values in nullables to explicit None (Python null) 
df = cast_dataframe_to_inferred_types(df, sample_size=10)

# Create the ephemeral GX context
context = gx.get_context()

# Add a pandas datasource
data_source = context.data_sources.add_pandas(name="pandas")

# Add a dataframe asset
data_asset = data_source.add_dataframe_asset(name="transactions_data")

# Define the batch (entire DataFrame)
batch_definition = data_asset.add_batch_definition_whole_dataframe(name="batch_def")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

# Create the expectation suite with a name
suite = gx.core.expectation_suite.ExpectationSuite(name="transactions_suite")

# Get the validator using the suite
validator = context.get_validator(batch=batch, expectation_suite=suite)

# check/coerce/apply expectations
apply_type_expectations(df, validator, sample_size=10)

# Add our own expectations
validator.expect_column_values_to_be_between("amount", min_value=0.01, max_value=100000)

# Validate
results = validator.validate()

# Print results
print(results)


ImportError: cannot import name 'cast_dataframe_to_inferred_types' from 'gx_type_inference' (/Volumes/Files/OneDrive/Projects/DM24/python-bank/gx_type_inference.py)